In [1]:
# ============================================================
# CUSTOMER SATISFACTION PREDICTION SYSTEM
# CLEAN & DEPLOYABLE VERSION
# ============================================================

# -----------------------------
# 1️⃣ Import Libraries
# -----------------------------
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# -----------------------------
# 2️⃣ Load Dataset
# -----------------------------
df = pd.read_csv("Customer_support_data.csv")
print("Dataset Loaded Successfully")
print("Shape:", df.shape)

# -----------------------------
# 3️⃣ Data Preprocessing
# -----------------------------

# Drop ID-like columns
id_cols = [c for c in df.columns if 'id' in c.lower() or 'uuid' in c.lower()]
df.drop(columns=id_cols, inplace=True, errors='ignore')

# Handle missing numeric values
num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].apply(lambda col: col.fillna(col.median()))

# Encode categorical features (low cardinality only)
cat_cols = df.select_dtypes(include='object').columns
low_cardinality = [c for c in cat_cols if df[c].nunique() < 20]
df = pd.get_dummies(df, columns=low_cardinality, drop_first=True)

# Remove remaining non-numeric columns
df.drop(columns=df.select_dtypes(exclude=[np.number]).columns, inplace=True, errors='ignore')

print("Preprocessing Completed")
print("Final Shape:", df.shape)

# -----------------------------
# 4️⃣ Target Variable Creation
# -----------------------------
df['CSAT_Category'] = (df['CSAT Score'] >= 4).astype(int)

X = df.drop(columns=['CSAT Score', 'CSAT_Category'])
y = df['CSAT_Category']

print("Features:", X.shape[1])
print("Target distribution:")
print(y.value_counts())

# -----------------------------
# 5️⃣ Train-Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------
# 6️⃣ Feature Scaling
# -----------------------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# -----------------------------
# 7️⃣ Train Final Model
# -----------------------------
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

model.fit(X_train, y_train)

print("Model Training Completed")

# -----------------------------
# 8️⃣ Model Evaluation
# -----------------------------
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("\nModel Accuracy:", round(accuracy, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# -----------------------------
# 9️⃣ Save Model & Scaler
# -----------------------------
joblib.dump(model, "model.pkl")
joblib.dump(scaler, "scaler.pkl")

print("\nModel and Scaler saved successfully")
print("Files created:")
print("- model.pkl")
print("- scaler.pkl")

# -----------------------------
# 10️⃣ Prediction Function (For Deployment)
# -----------------------------
def predict_customer_satisfaction(input_data):
    """
    input_data: dictionary with feature names as keys
    """
    input_df = pd.DataFrame([input_data])
    input_df = input_df.reindex(columns=X.columns, fill_value=0)
    scaled_input = scaler.transform(input_df)
    
    prediction = model.predict(scaled_input)[0]
    probability = model.predict_proba(scaled_input)[0][1]

    return {
        "Prediction": "Satisfied" if prediction == 1 else "Unsatisfied",
        "Confidence": round(probability, 3)
    }

print("\nDeployment-ready prediction function created")


Dataset Loaded Successfully
Shape: (85907, 20)
Preprocessing Completed
Final Shape: (85907, 3)
Features: 2
Target distribution:
CSAT_Category
1    70836
0    15071
Name: count, dtype: int64
Model Training Completed

Model Accuracy: 0.8266

Classification Report:
              precision    recall  f1-score   support

           0       0.36      0.00      0.01      2971
           1       0.83      1.00      0.90     14211

    accuracy                           0.83     17182
   macro avg       0.60      0.50      0.46     17182
weighted avg       0.75      0.83      0.75     17182


Model and Scaler saved successfully
Files created:
- model.pkl
- scaler.pkl

Deployment-ready prediction function created
